# Diagnosis and Procedure Code Pattern Prediction

This notebook develops and evaluates machine learning models that estimate the probability that selected procedure CCSR categories occur during an inpatient admission.<br>

The notebook builds on the feasibility analysis and focuses on three final targets:<br>

| Target | Role |
|---|---|
| MST019 | Primary modeling target |
| CAR024 | Financial opportunity target |
| GIS009 | Benchmark / upper-bound target |

The purpose is to produce probability estimates and operating thresholds that can later support review queue prioritization.

In [42]:
import os
from datetime import datetime
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from pycaret.classification import setup, compare_models, finalize_model
from sklearn.metrics import classification_report, roc_curve, roc_auc_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, log_loss, ConfusionMatrixDisplay
from sklearn.calibration import calibration_curve

pd.set_option('display.max_columns', None)

### Data directory setup
This notebook assumes that you can access the **Quantic Capstone** Google Drive set up for this project as a local path on your personal computer.<br>
Before running the notebook, set `base_directory` in the cell immediately below to the folder location on your machine.

If your local path is different, update `base_directory` so it points to your own accessible **Quantic Capstone** directory before running the next cell.

In [43]:
base_directory = "/Users/scott/Library/CloudStorage/GoogleDrive-clt.av8or@gmail.com/My Drive/Quantic Capstone"

In [44]:
# Working directory
working_dir=base_directory + '/Models/diagnosis_procedure_model/'

## Helper Functions<br>
### Display Results

In [45]:
# -------------------------------------------------------------------
# Report configuration
# -------------------------------------------------------------------
# Use of AI: ChatGPT was used to develop these functions. The first iterations were created
# for the Analytics Methods and Frameworks Project and enhanced for use in this notebook.
tgt_short="gis009" # Actual value gets set farther down in the code
html_report_title = f"{tgt_short.upper()} Model Selection and Threshold Optimization Results"
html_output_file_nm = f"model_results_{tgt_short}_{datetime.now().strftime('%Y%m%d_%H%M')}.html"
html_output_path = working_dir + html_output_file_nm
html_content = []


# -------------------------------------------------------------------
# Display + save helper
# -------------------------------------------------------------------
def display_and_save_df(
    df,
    title,
    format_dict=None,
    table_type=None
):
    global html_content

    df = df.copy()

    # -------------------------
    # Default format dictionaries
    # -------------------------
    if table_type == "model_scores":
        format_dict = {
            "Threshold": "{:.6f}",
            "Accuracy": "{:.3f}",
            "Precision": "{:.3f}",
            "Recall": "{:.3f}",
            "F1": "{:.3f}",
            "ROC-AUC": "{:.3f}",
            "Log Loss": "{:.3f}",
            "Flagged Cases": "{:,.0f}",
            "Flag Rate": "{:.3%}"
        }

    elif table_type == "threshold_table":
        format_dict = {
            "Threshold": "{:.6f}",
            "Accuracy": "{:.3f}",
            "Precision": "{:.3f}",
            "Recall": "{:.3f}",
            "F1": "{:.3f}",
            "Flagged Cases": "{:,.0f}",
            "Flag Rate": "{:.3%}",
            "F1 Rank": "{:,.0f}",
            "Precision Rank": "{:,.0f}",
            "Recall Rank": "{:,.0f}"
        }

    elif table_type == "threshold_compare":
        format_dict = {
            "Best Threshold": "{:.6f}",
            "Accuracy": "{:.3f}",
            "Precision": "{:.3f}",
            "Recall": "{:.3f}",
            "Best F1": "{:.3f}",
            "ROC-AUC": "{:.3f}",
            "Log Loss": "{:.3f}",
            "Flagged Cases": "{:,.0f}",
            "Flag Rate": "{:.3%}"
    }

    elif table_type == "feature_importance":
        format_dict = format_dict or {}
        for col in df.columns:
            if col != "Feature":
                format_dict.setdefault(col, "{:.6f}")

    # Auto-label confusion matrix
    if table_type == "confusion_matrix":
        df = pd.DataFrame(
            df,
            index=["Actual Negative", "Actual Positive"],
            columns=["Pred Negative", "Pred Positive"]
        )
        format_dict = {
            "Pred Negative": "{:,.0f}",
            "Pred Positive": "{:,.0f}"
        }

    if format_dict is None:
        format_dict = {}

    print(f"\n=== {title} ===")

    styled_df = df.style.format(format_dict)

    # -------------------------
    # Shared table styles
    # -------------------------
    table_styles = [
        {"selector": "th", "props": [("text-align", "left")]},
        {"selector": "td", "props": [("text-align", "right")]}
    ]

    # Left-align text columns when present
    left_align_cols = []
    for col in ["Model", "Model Type", "Feature"]:
        if col in df.columns:
            left_align_cols.append(col)

    if left_align_cols:
        styled_df = styled_df.set_properties(
            subset=left_align_cols,
            **{"text-align": "left"}
        )

    # Confusion matrix styling
    if table_type == "confusion_matrix":
        table_styles.extend([
            {"selector": "th.row_heading", "props": [("text-align", "left")]},
            {"selector": "th.col_heading", "props": [("text-align", "left")]}
        ])

    # Feature importance styling
    if table_type == "feature_importance":
        table_styles.extend([
            {
                "selector": "table",
                "props": [
                    ("width", "auto"),
                    ("display", "inline-table"),
                    ("table-layout", "auto")
                ]
            },
            {
                "selector": "th, td",
                "props": [
                    ("white-space", "nowrap"),
                    ("padding", "6px 10px")
                ]
            }
        ])

    styled_df = styled_df.set_table_styles(table_styles, overwrite=False)

    display(styled_df)

    table_classes = "feature-importance-table" if table_type == "feature_importance" else None

    html_table = styled_df.to_html(
        table_id=title.lower().replace(" ", "_"),
        border=1,
        classes=table_classes,
        escape=False
    )

    section = f"""
    <h2>{title}</h2>
    {html_table}
    <hr>
    """
    html_content.append(section)


# -------------------------------------------------------------------
# Finalize HTML report
# -------------------------------------------------------------------
def finalize_report(report_title=html_report_title, metadata=None):
    global html_content
    full_html = f"""<!DOCTYPE html>
<html>
<head>
    <title>{report_title}</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 40px; }}
        h1 {{ color: #2c3e50; }}
        h2 {{ color: #34495e; border-bottom: 2px solid #004C97; }}
        table {{ border-collapse: collapse; width: 100%; margin: 20px 0; }}
        .feature-importance-table {{ width: auto !important; display: inline-table; table-layout: auto; }}
        .feature-importance-table th,
        .feature-importance-table td {{ white-space: nowrap; padding: 6px 10px; }}
        th {{ text-align: left; }}
        th, td {{ border: 1px solid #ddd; padding: 12px; text-align: right; }}
        th {{ background-color: #004C97; color: white; }}
        tr:nth-child(even) {{ background-color: #f2f2f2; }}
        tr:hover {{ background-color: #e8f4f8; }}
        .metadata {{ font-size: 1.05em; color: #7f8c8d; margin: 10px 0 20px 0; }}
    </style>
</head>
<body>
    <h1>{report_title}</h1>
    <div class="metadata">
        <p><strong>Generated:</strong> {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
    </div>
""" + "".join(html_content) + """
    <hr>
    <p style="text-align: center; color: #7f8c8d;"><em>End of report</em></p>
</body>
</html>
"""

    with open(html_output_path, "w", encoding="utf-8") as f:
        f.write(full_html)

    print(f"\nOutput Saved: {html_output_path}")

## Model Scoring and Evaluation Functions

The following helper functions evaluate candidate models at both default and optimized thresholds. <br>
Outputs include accuracy, precision, recall, F1, ROC-AUC, log loss, flagged cases, and flag rate.

In [46]:
# Reusable functions to score models
# Use of AI: ChatGPT was used to develop these functions. The first iterations were created
# for the Analytics Methods and Frameworks Project and enhanced for use in this notebook.

# Determine model type from each of the candidate AutoML models
def get_model_type(model):

    if hasattr(model, 'best_estimator'):
        # FLAML
        return model.best_estimator
    elif hasattr(model, 'fitted_pipeline_'):
        # TPOT pipeline
        last_step = model.fitted_pipeline_.steps[-1][1]
        return last_step.__class__.__name__
    elif hasattr(model, 'named_steps'):
        # sklearn Pipeline
        last_step = list(model.named_steps.values())[-1]
        return last_step.__class__.__name__
    else:
        # Single estimator (PyCaret, etc.)
        return model.__class__.__name__

def eval_model(name, model, X_test, y_test, threshold=None):
    # Get model type automatically
    model_type = get_model_type(model)

    y_pred = model.predict(X_test)

    # Use predicted probabilities when available because downstream thresholding depends on probability scores.
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X_test)
        y_prob = (scores - scores.min()) / (scores.max() - scores.min())
    else:
        y_prob = y_pred

    if threshold is None:
        threshold = 0.5

    y_pred = (y_prob >= threshold).astype(int)

    results = {
        "Model": name,
        "Model Type": model_type,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Log Loss": log_loss(y_test, y_prob)
    }

    return results

### Data Prep

In [47]:
# Read input file
input_file_nm=base_directory + '/Models/feasibility_analysis/feasibility_model_input.csv'
df = pd.read_csv(input_file_nm)
# The feasibility input file contains all seven candidate targets.
# This notebook selects one target at a time for full model development.

In [48]:
# Define target columns
target_cols = [
'target_prccsr_car024',
'target_prccsr_esa001',
'target_prccsr_esa004',
'target_prccsr_gis009',
'target_prccsr_img008',
'target_prccsr_mst019',
'target_prccsr_res005'
]

## Target Selection

Set the `target` variable to one of the three final modeling targets. <br>
This notebook was run separately for MST019, CAR024, and GIS009 using the same feature set and evaluation workflow.

In [49]:
#target='target_prccsr_mst019'
#target='target_prccsr_car024'
target='target_prccsr_gis009'

# Will add the CCSR category code when saving the output
target_short = target.replace("target_prccsr_","").lower()
# Encode Gender
df['gender_female'] = (df['gender'] == 'F').astype(int)

# Convert DRG to string
df['drg_cd'] = df['drg_cd'].astype(str)


In [50]:
# Define X and y
X=df.drop(columns=target_cols + ['admit_id','gender','total_charge'])
y=df[target]

## Train/Test Split

The dataset is split into training and test sets using stratification to preserve the target prevalence in both samples.<br>
The test set is held out for model evaluation.

In [51]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [52]:
# PyCaret expects the training features and target in a single dataframe.
pc_df = X_train.copy()
pc_df[target] = y_train

In [53]:
numeric_cols=[
    'age',
    'rom_score',
    'soi_score',
    'length_of_stay',
    'diagnosis_count',
    'distinct_ccsr_count',
    'secondary_ccsr_count']


## Candidate Model Development

Two AutoML approaches are evaluated:

- **PyCaret calibrated model:** supports model comparison, calibration, and probability estimation.
- **FLAML:** performs efficient automated model selection and hyperparameter optimization.

Both models are evaluated using the same train/test split so their performance can be compared directly.
### PyCaret

In [54]:
# Train PyCaret Model
clf = setup(
    data=pc_df,
    target=target,
    session_id=42,
    numeric_features=numeric_cols,
    categorical_features=['drg_cd'],
    normalize=True,
    normalize_method='zscore',
    fold=5,
    verbose=False,
    html=False)

pc_best_model = compare_models(
    sort="AUC",
    exclude=["ridge"],
    verbose=False)

pc_final_model = finalize_model(pc_best_model)

pc_calibrated_model = CalibratedClassifierCV(
    pc_final_model,
    method='sigmoid',
    cv=5)

pc_calibrated_model.fit(X_train, y_train)

CalibratedClassifierCV(cv=5, ensemble=True,
                       estimator=Pipeline(memory=Memory(location=None),
                                          steps=[('numerical_imputer',
                                                  TransformerWrapper(exclude=None,
                                                                     include=['age',
                                                                              'rom_score',
                                                                              'soi_score',
                                                                              'length_of_stay',
                                                                              'diagnosis_count',
                                                                              'distinct_ccsr_count',
                                                                              'secondary_ccsr_count'],
                                                                     transformer=SimpleImputer(add_indicator=False,
                                                                                               copy=True,
                                                                                               fill_value=None,
                                                                                               keep_emp...
                                                                interaction_constraints=None,
                                                                learning_rate=None,
                                                                max_bin=None,
                                                                max_cat_threshold=None,
                                                                max_cat_to_onehot=None,
                                                                max_delta_step=None,
                                                                max_depth=None,
                                                                max_leaves=None,
                                                                min_child_weight=None,
                                                                missing=nan,
                                                                monotone_constraints=None,
                                                                multi_strategy=None,
                                                                n_estimators=None,
                                                                n_jobs=-1,
                                                                num_parallel_tree=None,
                                                                objective='binary:logistic', ...))],
                                          verbose=False),
                       method='sigmoid', n_jobs=None)

In [55]:
pc_results = eval_model("PyCaret (Calibrated)", pc_calibrated_model, X_test, y_test)

In [56]:
pc_results_df = pd.DataFrame([pc_results])

In [57]:
display_and_save_df(
    pc_results_df,
    "Candidate Model Scores Using Default Threshold",
    table_type="model_scores")


=== Candidate Model Scores Using Default Threshold ===


,Model,Model Type,Threshold,Accuracy,Precision,Recall,F1,ROC-AUC,Log Loss
0,PyCaret (Calibrated),CalibratedClassifierCV,0.500000,0.991,0.368,0.078,0.129,0.983,0.033


### FLAML

In [58]:
# FLAML
from flaml import AutoML

flaml_model = AutoML()
flaml_settings = {
    "time_budget": 600,
    "metric": 'roc_auc',
    "task": 'classification',
    "estimator_list": ["lgbm", "xgboost", "xgb_limitdepth", "rf", "extra_tree"],
    "verbose": 1,
    "eval_method": "cv", # use cross-validation
    "n_splits": 5,
    "seed": 42,
}
flaml_model.fit(X_train=X_train, y_train=y_train, **flaml_settings)
flaml_results = eval_model("FLAML", flaml_model, X_test, y_test)

print("Best estimator name:", flaml_model.best_estimator)
print("Best hyperparameters:", flaml_model.best_config)
print("Best validation loss:", flaml_model.best_loss)

Best estimator name: xgboost
Best hyperparameters: {'n_estimators': 1414, 'max_leaves': 13, 'min_child_weight': 0.9069456043538519, 'learning_rate': 0.07575193133929094, 'subsample': 0.6797111342353143, 'colsample_bylevel': 0.9125101115227117, 'colsample_bytree': 1.0, 'reg_alpha': 0.0212742822119279, 'reg_lambda': 314.28203848879633}
Best validation loss: 0.017295548621949486


Results - target='target_prccsr_mst019' <br>
Best estimator name: xgb_limitdepth <br>
Best hyperparameters: {'n_estimators': 1028, 'max_depth': 6, 'min_child_weight': 1.4640336660392816, 'learning_rate': 0.02159558962109372, 'subsample': 0.8640052147174596, 'colsample_bylevel': 0.8726369812699186, 'colsample_bytree': 0.9407746256493329, 'reg_alpha': 0.0009765625, 'reg_lambda': 13.44701768864806} <br>
Best validation loss: 0.07071096376772872

In [59]:
# Display Results
flaml_results_df = pd.DataFrame([flaml_results])
display_and_save_df(
    flaml_results_df,
    "Candidate Model Score Using Default Threshold",
    table_type="model_scores")


=== Candidate Model Score Using Default Threshold ===


,Model,Model Type,Threshold,Accuracy,Precision,Recall,F1,ROC-AUC,Log Loss
0,FLAML,xgboost,0.500000,0.991,0.465,0.040,0.073,0.984,0.024


In [60]:
# Review Both Candidate Models
candidate_results_df = pd.DataFrame([pc_results, flaml_results ])
candidate_results_df = candidate_results_df.sort_values(by="ROC-AUC", ascending=False)

display_and_save_df(
    candidate_results_df,
    "Candidate Model Scores Using Default Threshold",
    table_type="model_scores")


=== Candidate Model Scores Using Default Threshold ===


,Model,Model Type,Threshold,Accuracy,Precision,Recall,F1,ROC-AUC,Log Loss
1,FLAML,xgboost,0.500000,0.991,0.465,0.040,0.073,0.984,0.024
0,PyCaret (Calibrated),CalibratedClassifierCV,0.500000,0.991,0.368,0.078,0.129,0.983,0.033


Interpretation:<br>
If we only cared about the models at the default threshold, these models are essentially neck-and-neck.<br>
But with a Recall of less than 1%, threshold optimization is required.

In [61]:
# Threshold Optimization
# Use of AI: ChatGPT was used extensively to troubleshoot and develop the final versions of these functions to
# properly deal with the different attributes returned by each of the candidate models.
# The first iterations were created # for the Analytics Methods and Frameworks Project
# and enhanced for use in this notebook.

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,
    precision_recall_curve
)

def evaluate_predictions(y_true, y_prob, threshold=0.5, model_name=None):
    """
    Evaluate predicted probabilities at a specified classification threshold.
    Returns:
        - results_dict: one-row summary dict
        - cm: confusion matrix ndarray
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    results_dict = {
        "Model": model_name if model_name is not None else "Model",
        "Model Type": "Probability Classifier",
        "Threshold": float(threshold),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "Log Loss": log_loss(y_true, y_prob),
        "Flagged Cases": int(y_pred.sum()),
        "Flag Rate": float(y_pred.mean())
    }

    cm = confusion_matrix(y_true, y_pred)
    return results_dict, cm


def build_threshold_table(y_true, y_prob, thresholds=None):
    """
    Build a threshold sweep table using explicit thresholds.
    This is often easier to interpret than the raw thresholds returned by
    precision_recall_curve().
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    if thresholds is None:
        thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

    rows = []

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        acc = accuracy_score(y_true, y_pred)

        rows.append({
            "Threshold": float(t),
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "Flagged Cases": int(y_pred.sum()),
            "Flag Rate": float(y_pred.mean())
        })

    threshold_df = pd.DataFrame(rows)

    # Add rank columns for quick inspection if helpful
    threshold_df["F1 Rank"] = threshold_df["F1"].rank(method="dense", ascending=False).astype(int)
    threshold_df["Precision Rank"] = threshold_df["Precision"].rank(method="dense", ascending=False).astype(int)
    threshold_df["Recall Rank"] = threshold_df["Recall"].rank(method="dense", ascending=False).astype(int)

    return threshold_df.sort_values("Threshold").reset_index(drop=True)


def optimize_threshold(
    y_true,
    y_prob,
    metric="f1",
    thresholds=None,
    min_precision=None,
    min_recall=None
):
    """
    Optimize threshold using a threshold sweep table.

    Parameters
    ----------
    metric : str
        One of: 'f1', 'precision', 'recall'
    min_precision : float or None
        Optional minimum precision constraint.
    min_recall : float or None
        Optional minimum recall constraint.

    Returns
    -------
    best_threshold : float
    threshold_df : DataFrame
    best_row : Series
    """
    threshold_df = build_threshold_table(y_true, y_prob, thresholds=thresholds).copy()

    candidate_df = threshold_df.copy()

    if min_precision is not None:
        candidate_df = candidate_df[candidate_df["Precision"] >= min_precision]

    if min_recall is not None:
        candidate_df = candidate_df[candidate_df["Recall"] >= min_recall]

    # Fallback if constraints are too strict
    if candidate_df.empty:
        candidate_df = threshold_df.copy()

    metric_map = {
        "f1": "F1",
        "precision": "Precision",
        "recall": "Recall"
    }

    if metric not in metric_map:
        raise ValueError("metric must be one of: 'f1', 'precision', 'recall'")

    score_col = metric_map[metric]

    # Tie-breakers:
    # 1) maximize selected metric
    # 2) maximize precision
    # 3) maximize recall
    # 4) choose the higher threshold to stay conservative if still tied
    candidate_df = candidate_df.sort_values(
        by=[score_col, "Precision", "Recall", "Threshold"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    best_row = candidate_df.iloc[0]
    best_threshold = float(best_row["Threshold"])

    return best_threshold, threshold_df, best_row


def evaluate_model_at_default_and_optimized_threshold(
    model_name,
    y_true,
    y_prob,
    optimization_metric="f1",
    thresholds=None,
    min_precision=None,
    min_recall=None
):
    """
    Main wrapper to produce:
      - default-threshold results row
      - optimized-threshold results row
      - threshold sweep table
      - default confusion matrix
      - optimized confusion matrix
      - best threshold metadata
    """
    default_results, default_cm = evaluate_predictions(
        y_true=y_true,
        y_prob=y_prob,
        threshold=0.5,
        model_name=model_name
    )
    default_results["Model Type"] = "Default Threshold"

    best_threshold, threshold_df, best_row = optimize_threshold(
        y_true=y_true,
        y_prob=y_prob,
        metric=optimization_metric,
        thresholds=thresholds,
        min_precision=min_precision,
        min_recall=min_recall
    )

    optimized_results, optimized_cm = evaluate_predictions(
        y_true=y_true,
        y_prob=y_prob,
        threshold=best_threshold,
        model_name=model_name
    )
    optimized_results["Model Type"] = f"Optimized Threshold ({optimization_metric.upper()})"

    summary_df = pd.DataFrame([default_results, optimized_results])

    return {
        "summary_df": summary_df,
        "threshold_df": threshold_df,
        "default_cm": default_cm,
        "optimized_cm": optimized_cm,
        "best_threshold": best_threshold,
        "best_row": best_row
    }

In [62]:
pycaret_prob = pc_calibrated_model.predict_proba(X_test)[:, 1]
flaml_prob = flaml_model.predict_proba(X_test)[:, 1]

pycaret_eval = evaluate_model_at_default_and_optimized_threshold(
    model_name="PyCaret (Calibrated)",
    y_true=y_test,
    y_prob=pycaret_prob,
    optimization_metric="f1"
)

flaml_eval = evaluate_model_at_default_and_optimized_threshold(
    model_name="FLAML",
    y_true=y_test,
    y_prob=flaml_prob,
    optimization_metric="f1"
)

results_df = pd.concat(
    [pycaret_eval["summary_df"], flaml_eval["summary_df"]],
    ignore_index=True
)

threshold_compare_df = pd.DataFrame([
    {
        "Model": "PyCaret (Calibrated)",
        "Best Threshold": pycaret_eval["best_threshold"],
        "Accuracy": pycaret_eval["best_row"]["Accuracy"],
        "Precision": pycaret_eval["best_row"]["Precision"],
        "Recall": pycaret_eval["best_row"]["Recall"],
        "F1": pycaret_eval["best_row"]["F1"],
        "ROC-AUC": pycaret_eval["summary_df"].iloc[1]["ROC-AUC"],
        "Log Loss": pycaret_eval["summary_df"].iloc[1]["Log Loss"],
        "Flagged Cases": pycaret_eval["best_row"]["Flagged Cases"],
        "Flag Rate": pycaret_eval["best_row"]["Flag Rate"]
    },
    {
        "Model": "FLAML",
        "Best Threshold": flaml_eval["best_threshold"],
        "Accuracy": flaml_eval["best_row"]["Accuracy"],
        "Precision": flaml_eval["best_row"]["Precision"],
        "Recall": flaml_eval["best_row"]["Recall"],
        "F1": flaml_eval["best_row"]["F1"],
        "ROC-AUC": flaml_eval["summary_df"].iloc[1]["ROC-AUC"],
        "Log Loss": flaml_eval["summary_df"].iloc[1]["Log Loss"],
        "Flagged Cases": flaml_eval["best_row"]["Flagged Cases"],
        "Flag Rate": flaml_eval["best_row"]["Flag Rate"]
    }
])

In [63]:
display_and_save_df(
    results_df,
    title="Model Scores: Default vs Optimized Threshold",
    table_type="model_scores"
)


=== Model Scores: Default vs Optimized Threshold ===


,Model,Model Type,Threshold,Accuracy,Precision,Recall,F1,ROC-AUC,Log Loss,Flagged Cases,Flag Rate
0,PyCaret (Calibrated),Default Threshold,0.500000,0.991,0.368,0.078,0.129,0.983,0.033,315,0.187%
1,PyCaret (Calibrated),Optimized Threshold (F1),0.070000,0.984,0.280,0.495,0.357,0.983,0.033,"2,627",1.563%
2,FLAML,Default Threshold,0.500000,0.991,0.465,0.040,0.073,0.984,0.024,127,0.076%
3,FLAML,Optimized Threshold (F1),0.160000,0.982,0.269,0.570,0.365,0.984,0.024,"3,152",1.875%


In [64]:
display_and_save_df(
    threshold_compare_df,
    title="Best Threshold Summary by Model",
    table_type="threshold_compare"
)



=== Best Threshold Summary by Model ===


,Model,Best Threshold,Accuracy,Precision,Recall,F1,ROC-AUC,Log Loss,Flagged Cases,Flag Rate
0,PyCaret (Calibrated),0.070000,0.984,0.280,0.495,0.357403,0.983,0.033,"2,627",1.563%
1,FLAML,0.160000,0.982,0.269,0.570,0.365244,0.984,0.024,"3,152",1.875%


## Initial Model Interpretation

Both candidate models demonstrate strong performance after threshold optimization. <br>
The results suggest that PyCaret and FLAML are learning similar diagnosis-procedure signal from the feature set.<br>

In general:
- FLAML produced slightly stronger precision and F1 performance.
- PyCaret produced slightly higher recall in some cases.
- Both models produced usable probability estimates for downstream review prioritization.

Final model selection is based on validation performance rather than test-set threshold optimization alone.

In [65]:
display_and_save_df(
    pycaret_eval["threshold_df"].sort_values("F1", ascending=False).head(15),
    title="PyCaret Threshold Sweep (Top 15 by F1)",
    table_type="threshold_table"
)



=== PyCaret Threshold Sweep (Top 15 by F1) ===


,Threshold,Accuracy,Precision,Recall,F1,Flagged Cases,Flag Rate,F1 Rank,Precision Rank,Recall Rank
2,0.070000,0.984,0.280,0.495,0.357,"2,627",1.563%,1,84,3
1,0.060000,0.983,0.271,0.522,0.357,"2,862",1.703%,2,85,2
0,0.050000,0.982,0.263,0.552,0.356,"3,121",1.857%,3,86,1
3,0.080000,0.985,0.286,0.468,0.355,"2,435",1.449%,4,83,4
4,0.090000,0.986,0.290,0.441,0.350,"2,258",1.343%,5,82,5
5,0.100000,0.986,0.296,0.425,0.349,"2,136",1.271%,6,81,6
7,0.120000,0.987,0.311,0.395,0.348,"1,890",1.124%,7,79,8
6,0.110000,0.986,0.303,0.407,0.348,"1,996",1.187%,8,80,7
8,0.130000,0.987,0.312,0.376,0.341,"1,789",1.064%,9,78,9
9,0.140000,0.987,0.314,0.358,0.334,"1,695",1.008%,10,77,10


In [66]:
display_and_save_df(
    flaml_eval["threshold_df"].sort_values("F1", ascending=False).head(15),
    title="FLAML Threshold Sweep (Top 15 by F1)",
    table_type="threshold_table"
)


=== FLAML Threshold Sweep (Top 15 by F1) ===


,Threshold,Accuracy,Precision,Recall,F1,Flagged Cases,Flag Rate,F1 Rank,Precision Rank,Recall Rank
11,0.160000,0.982,0.269,0.570,0.365,"3,152",1.875%,1,57,12
13,0.180000,0.984,0.283,0.515,0.365,"2,709",1.612%,2,55,14
12,0.170000,0.983,0.274,0.539,0.363,"2,923",1.739%,3,56,13
9,0.140000,0.980,0.254,0.629,0.362,"3,681",2.190%,4,59,10
10,0.150000,0.981,0.259,0.592,0.361,"3,396",2.020%,5,58,11
16,0.210000,0.986,0.306,0.437,0.360,"2,121",1.262%,6,52,17
14,0.190000,0.985,0.287,0.482,0.360,"2,497",1.485%,7,54,15
8,0.130000,0.979,0.246,0.661,0.359,"3,986",2.371%,8,60,9
17,0.220000,0.987,0.313,0.417,0.358,"1,976",1.176%,9,51,18
15,0.200000,0.986,0.294,0.456,0.357,"2,303",1.370%,10,53,16


In [67]:
display_and_save_df(
    pycaret_eval["default_cm"],
    title="PyCaret Confusion Matrix at Threshold 0.5",
    table_type="confusion_matrix"
)


=== PyCaret Confusion Matrix at Threshold 0.5 ===


,Pred Negative,Pred Positive
Actual Negative,"166,409",199
Actual Positive,"1,370",116


In [68]:
display_and_save_df(
    pycaret_eval["optimized_cm"],
    title=f"PyCaret Confusion Matrix at Optimized Threshold ({pycaret_eval['best_threshold']:.6f})",
    table_type="confusion_matrix"
)


=== PyCaret Confusion Matrix at Optimized Threshold (0.070000) ===


,Pred Negative,Pred Positive
Actual Negative,"164,716","1,892"
Actual Positive,751,735


In [69]:
display_and_save_df(
    flaml_eval["default_cm"],
    title="FLAML Confusion Matrix at Threshold 0.5",
    table_type="confusion_matrix"
)


=== FLAML Confusion Matrix at Threshold 0.5 ===


,Pred Negative,Pred Positive
Actual Negative,"166,540",68
Actual Positive,"1,427",59


In [70]:
display_and_save_df(
    flaml_eval["optimized_cm"],
    title=f"FLAML Confusion Matrix at Optimized Threshold ({flaml_eval['best_threshold']:.6f})",
    table_type="confusion_matrix"
)


=== FLAML Confusion Matrix at Optimized Threshold (0.160000) ===


,Pred Negative,Pred Positive
Actual Negative,"164,303","2,305"
Actual Positive,639,847


Preliminary Model Selection<br>
Both models demonstrated nearly identical discrimination and threshold-optimized performance. FLAML was selected as the final model due to slightly better probability calibration (lower log loss), which improves the reliability of downstream thresholding and financial opportunity estimation.<br>
**Important**
The threshold is optimized for F1, in subsequent steps, we'll look at optimizing for financial impact and for case review.<>
As of now, we have a **baseline operating threshold**.

## Validation-Based Threshold Selection

The initial threshold sweep identifies useful operating points, but optimizing thresholds directly on the test set can overstate performance.<br>
To avoid this, the original training data is split again into a training subset and validation subset.<br>

Models are retrained on the reduced training subset, thresholds are selected using validation data, and final performance is compared against the held-out test results.<br>
This creates a more defensible operating threshold for downstream financial and review queue analysis.

In [71]:
# Split the original training set again
X_train2, X_val, y_train2, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.30,
    stratify=y_train,
    random_state=42
)

print("X_train2 shape:", X_train2.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)
print("Positive rate - train2:", y_train2.mean())
print("Positive rate - val:", y_val.mean())
print("Positive rate - test:", y_test.mean())

X_train2 shape: (274551, 31)
X_val shape: (117666, 31)
X_test shape: (168094, 31)
Positive rate - train2: 0.008839887671143066
Positive rate - val: 0.008838576989104754
Positive rate - test: 0.008840291741525575


In [72]:
# Train pyCaret using X_train2
pc_df_val = X_train2.copy()
pc_df_val[target] = y_train2

numeric_cols_val=[
    'age',
    'rom_score',
    'soi_score',
    'length_of_stay',
    'diagnosis_count',
    'distinct_ccsr_count',
    'secondary_ccsr_count']



clf = setup(
    data=pc_df_val,
    target=target,
    session_id=42,
    fold=5,
    numeric_features=numeric_cols_val,
    categorical_features=['drg_cd'],
    normalize=True,
    normalize_method='zscore',
    verbose=False,
    html=False
)

pc_best_model_val = compare_models(
    sort='AUC',
    exclude=['ridge'],
    verbose=False
)

pc_final_model_val = finalize_model(pc_best_model_val)

pc_calibrated_model_val = CalibratedClassifierCV(
    estimator=pc_final_model_val,
    method='sigmoid',
    cv=5
)

pc_calibrated_model_val.fit(X_train2, y_train2)

CalibratedClassifierCV(cv=5, ensemble=True,
                       estimator=Pipeline(memory=Memory(location=None),
                                          steps=[('numerical_imputer',
                                                  TransformerWrapper(exclude=None,
                                                                     include=['age',
                                                                              'rom_score',
                                                                              'soi_score',
                                                                              'length_of_stay',
                                                                              'diagnosis_count',
                                                                              'distinct_ccsr_count',
                                                                              'secondary_ccsr_count'],
                                                                     transformer=SimpleImputer(add_indicator=False,
                                                                                               copy=True,
                                                                                               fill_value=None,
                                                                                               keep_emp...
                                                                interaction_constraints=None,
                                                                learning_rate=None,
                                                                max_bin=None,
                                                                max_cat_threshold=None,
                                                                max_cat_to_onehot=None,
                                                                max_delta_step=None,
                                                                max_depth=None,
                                                                max_leaves=None,
                                                                min_child_weight=None,
                                                                missing=nan,
                                                                monotone_constraints=None,
                                                                multi_strategy=None,
                                                                n_estimators=None,
                                                                n_jobs=-1,
                                                                num_parallel_tree=None,
                                                                objective='binary:logistic', ...))],
                                          verbose=False),
                       method='sigmoid', n_jobs=None)

In [73]:
pc_results_val=eval_model("PyCaret (Calibrated) - Validation", pc_calibrated_model_val,X_val,y_val)
pc_results_val_df=pd.DataFrame([pc_results_val])
display_and_save_df(pc_results_val_df,"Candidate Model Scores using Validation Data Set - Default Threshold",table_type="model_scores")


=== Candidate Model Scores using Validation Data Set - Default Threshold ===


,Model,Model Type,Threshold,Accuracy,Precision,Recall,F1,ROC-AUC,Log Loss
0,PyCaret (Calibrated) - Validation,CalibratedClassifierCV,0.500000,0.991,0.368,0.068,0.115,0.982,0.034


In [74]:
# Train FLAML on X_train2
flaml_model_val = AutoML()

flaml_settings = {
    "time_budget": 600,
    "metric": "roc_auc",
    "task": "classification",
    "verbose": 1,
    "eval_method": "cv",
    "n_splits": 5,
    "seed": 42,
    "estimator_list": ["lgbm", "xgboost", "xgb_limitdepth", "rf", "extra_tree"]
}

flaml_model_val.fit(
    X_train=X_train2,
    y_train=y_train2,
    **flaml_settings
)

print("Best estimator name:", flaml_model_val.best_estimator)
print("Best hyperparameters:", flaml_model_val.best_config)
print("Best validation loss:", flaml_model_val.best_loss)

Best estimator name: xgboost
Best hyperparameters: {'n_estimators': 1414, 'max_leaves': 13, 'min_child_weight': 0.9069456043538519, 'learning_rate': 0.07575193133929094, 'subsample': 0.6797111342353143, 'colsample_bylevel': 0.9125101115227117, 'colsample_bytree': 1.0, 'reg_alpha': 0.0212742822119279, 'reg_lambda': 314.28203848879633}
Best validation loss: 0.01757046761133487


In [75]:
flaml_results_val = eval_model("FLAML", flaml_model_val, X_val, y_val)
flaml_results_val_df=pd.DataFrame([flaml_results_val])
display_and_save_df(flaml_results_val_df,"Candidate Model Scores using Validation Data Set - Default Threshold", table_type="model_scores")


=== Candidate Model Scores using Validation Data Set - Default Threshold ===


,Model,Model Type,Threshold,Accuracy,Precision,Recall,F1,ROC-AUC,Log Loss
0,FLAML,xgboost,0.500000,0.991,0.396,0.035,0.064,0.983,0.025


In [76]:
# Optimize Thresholds on X_val
# Use predicted probabilities from the validation set, not the test set
pycaret_val_prob = pc_calibrated_model_val.predict_proba(X_val)[:, 1]
flaml_val_prob = flaml_model_val.predict_proba(X_val)[:, 1]

pycaret_val_eval = evaluate_model_at_default_and_optimized_threshold(
    model_name="PyCaret (Calibrated)",
    y_true=y_val,
    y_prob=pycaret_val_prob,
    optimization_metric="f1"
)

flaml_val_eval = evaluate_model_at_default_and_optimized_threshold(
    model_name="FLAML",
    y_true=y_val,
    y_prob=flaml_val_prob,
    optimization_metric="f1"
)

val_results_df = pd.concat(
    [pycaret_val_eval["summary_df"], flaml_val_eval["summary_df"]],
    ignore_index=True
)

val_threshold_compare_df = pd.DataFrame([
    {
        "Model": "PyCaret (Calibrated)-Validation Data Set",
        "Best Threshold": pycaret_val_eval["best_threshold"],
        "Accuracy": pycaret_val_eval["best_row"]["Accuracy"],
        "Precision": pycaret_val_eval["best_row"]["Precision"],
        "Recall": pycaret_val_eval["best_row"]["Recall"],
        "F1": pycaret_val_eval["best_row"]["F1"],
        "ROC-AUC": pycaret_val_eval["summary_df"].iloc[1]["ROC-AUC"],
        "Log Loss": pycaret_val_eval["summary_df"].iloc[1]["Log Loss"],
        "Flagged Cases": pycaret_val_eval["best_row"]["Flagged Cases"],
        "Flag Rate": pycaret_val_eval["best_row"]["Flag Rate"]
    },
    {
        "Model": "FLAML-Validation Data Set",
        "Best Threshold": flaml_val_eval["best_threshold"],
        "Accuracy": flaml_val_eval["best_row"]["Accuracy"],
        "Precision": flaml_val_eval["best_row"]["Precision"],
        "Recall": flaml_val_eval["best_row"]["Recall"],
        "F1": flaml_val_eval["best_row"]["F1"],
        "ROC-AUC": flaml_val_eval["summary_df"].iloc[1]["ROC-AUC"],
        "Log Loss": flaml_val_eval["summary_df"].iloc[1]["Log Loss"],
        "Flagged Cases": flaml_val_eval["best_row"]["Flagged Cases"],
        "Flag Rate": flaml_val_eval["best_row"]["Flag Rate"]
    }
])


In [77]:
display_and_save_df(
    val_threshold_compare_df,
    title="Best Threshold Summary by Model - Validation Data Set",
    table_type="threshold_compare"
)


=== Best Threshold Summary by Model - Validation Data Set ===


,Model,Best Threshold,Accuracy,Precision,Recall,F1,ROC-AUC,Log Loss,Flagged Cases,Flag Rate
0,PyCaret (Calibrated)-Validation Data Set,0.060000,0.983,0.261,0.486,0.339496,0.982,0.034,"1,935",1.644%
1,FLAML-Validation Data Set,0.150000,0.981,0.253,0.575,0.351042,0.983,0.025,"2,367",2.012%


In [78]:
# Summarize the results
model_summary_df=pd.concat([threshold_compare_df, val_threshold_compare_df],ignore_index=True)
model_summary_df.sort_values(by=["Model"], inplace=True)
display_and_save_df(
    model_summary_df,
    title="Best Threshold Summary by Model - Test Data Set vs. Validation Data Set",
    table_type="threshold_compare"
)


=== Best Threshold Summary by Model - Test Data Set vs. Validation Data Set ===


,Model,Best Threshold,Accuracy,Precision,Recall,F1,ROC-AUC,Log Loss,Flagged Cases,Flag Rate
1,FLAML,0.160000,0.982,0.269,0.570,0.365244,0.984,0.024,"3,152",1.875%
3,FLAML-Validation Data Set,0.150000,0.981,0.253,0.575,0.351042,0.983,0.025,"2,367",2.012%
0,PyCaret (Calibrated),0.070000,0.984,0.280,0.495,0.357403,0.983,0.033,"2,627",1.563%
2,PyCaret (Calibrated)-Validation Data Set,0.060000,0.983,0.261,0.486,0.339496,0.982,0.034,"1,935",1.644%


## Manual Model Selection Checkpoint

This notebook intentionally pauses after the test and validation summaries are generated.<br>
At this point, the analyst reviews model performance, selects the final model, and confirms the operating threshold before saving the model artifact.

For this project, the final saved model uses:
- the model trained on the full training dataset, and
- the threshold selected from validation performance.

In [79]:
# Introduce a deliberate breakpoint to evaluate the candidate model scores and select the final one.
raise SystemExit("Review results, adjust variables, then continue manually.")

SystemExit: Review results, adjust variables, then continue manually.

In [80]:
# Select the final model
# Use the model trained on the full training dataset,
# and the threshold calculated from the validation dataset.

# If FLAML is the winner:
final_model_type="flaml"
final_model = flaml_model
final_threshold = flaml_val_eval["best_threshold"]
final_y_prob = final_model.predict_proba(X_test)[:, 1]
final_y_pred = (final_y_prob >= final_threshold).astype(int)

# If PyCaret is the winner:
#final_model_type="pycaret_calibrated"
#final_model = pc_final_model
#final_threshold = pycaret_val_eval["best_threshold"]
#final_y_prob = final_model.predict_proba(X_test)[:, 1]
#final_y_pred = (final_y_prob >= final_threshold).astype(int)

print(f"Model Selected: {final_model_type}")
print(f"Operating Threshold: {final_threshold:.6f}")
print("Predicted positives:", final_y_pred.sum())
print(f"Flag rate: {final_y_pred.mean():.6f}")

Model Selected: flaml
Operating Threshold: 0.150000
Predicted positives: 3396
Flag rate: 0.020203


## Save Final Model Artifact

The saved model artifact includes the fitted model, selected validation-derived threshold, target name, model type, feature list, and final prediction outputs.<br>
This artifact is used later to score MIMIC proof-of-concept admissions and generate review queues.

In [81]:
# Save Final Model
import joblib


model_artifacts = {
    "model": final_model,
    "threshold": final_threshold,
    "y_prob": final_y_prob,
    "y_pred": final_y_pred,
    "target": target,
    "model_type": final_model_type,
    "features": list(X_test.columns)
}
model_output_file_nm = f"{final_model_type}_{target_short}_model.pkl"
model_output_path = os.path.join(working_dir,model_output_file_nm)

joblib.dump(model_artifacts, model_output_path)

print(f"Model saved to: {model_output_path}")

Model saved to: /Users/scott/Library/CloudStorage/GoogleDrive-clt.av8or@gmail.com/My Drive/Quantic Capstone/Models/diagnosis_procedure_model/flaml_gis009_model.pkl


In [82]:
finalize_report()


Output Saved: /Users/scott/Library/CloudStorage/GoogleDrive-clt.av8or@gmail.com/My Drive/Quantic Capstone/Models/diagnosis_procedure_model/model_results_gis009_20260614_2100.html
